In [1]:
import pandas as pd
import numpy as np

In [2]:
# Loading our new large dataset:
df = pd.read_csv("../data/raw/datapulse_sales_large.csv")

In [3]:
df.head()

,Order_ID,Order_Date,Customer_ID,Product,Category,Region,Quantity,Unit_Price,Discount,Payment_Method,Returned,Gross_Sales,Discount_Amount,Net_Sales
0,O00001,2025-01-01 00:00:00.000000000,C0017,Headphones,Accessories,North,3,2500,0.00,Credit Card,No,7500,0.0,7500.0
1,O00002,2025-01-01 02:36:59.483896779,C0056,Monitor,Electronics,East,4,15000,0.00,Credit Card,No,60000,0.0,60000.0
2,O00003,2025-01-01 05:13:58.967793558,C0182,Laptop,Electronics,North,2,65000,0.05,Cash,No,130000,6500.0,123500.0
3,O00004,2025-01-01 07:50:58.451690338,C0144,Laptop,Electronics,East,3,65000,0.10,Debit Card,No,195000,19500.0,175500.0
4,O00005,2025-01-01 10:27:57.935587117,C0319,Laptop,Electronics,West,1,65000,0.00,Debit Card,No,65000,0.0,65000.0


In [4]:
# First: performing a quality audit
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Rows: 5000
Columns: 14


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order_ID         5000 non-null   object 
 1   Order_Date       5000 non-null   object 
 2   Customer_ID      5000 non-null   object 
 3   Product          5000 non-null   object 
 4   Category         5000 non-null   object 
 5   Region           5000 non-null   object 
 6   Quantity         5000 non-null   int64  
 7   Unit_Price       5000 non-null   int64  
 8   Discount         5000 non-null   float64
 9   Payment_Method   5000 non-null   object 
 10  Returned         5000 non-null   object 
 11  Gross_Sales      5000 non-null   int64  
 12  Discount_Amount  5000 non-null   float64
 13  Net_Sales        5000 non-null   float64
dtypes: float64(3), int64(3), object(8)
memory usage: 547.0+ KB


In [6]:
df.isnull().sum()

Order_ID           0
Order_Date         0
Customer_ID        0
Product            0
Category           0
Region             0
Quantity           0
Unit_Price         0
Discount           0
Payment_Method     0
Returned           0
Gross_Sales        0
Discount_Amount    0
Net_Sales          0
dtype: int64

In [7]:
df.duplicated().sum()

np.int64(0)

In [8]:
df.describe()

,Quantity,Unit_Price,Discount,Gross_Sales,Discount_Amount,Net_Sales
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000
mean,2.990200,20152.180000,0.075020,60282.340000,4463.480000,55818.860000
std,1.421305,21957.747238,0.056175,77995.546663,8041.222552,72446.269532
min,1.000000,800.000000,0.000000,800.000000,0.000000,680.000000
25%,2.000000,2500.000000,0.000000,6000.000000,0.000000,5400.000000
50%,3.000000,15000.000000,0.100000,17500.000000,750.000000,15000.000000
75%,4.000000,40000.000000,0.150000,90000.000000,6000.000000,80000.000000
max,5.000000,65000.000000,0.150000,325000.000000,48750.000000,325000.000000


In [9]:
# Check Unique Values
print("Products:")
print(df["Product"].unique())

print("\nCategories:")
print(df["Category"].unique())

print("\nRegions:")
print(df["Region"].unique())

print("\nPayment Methods:")
print(df["Payment_Method"].unique())

print("\nReturned:")
print(df["Returned"].unique())

Products:
['Headphones' 'Monitor' 'Laptop' 'Keyboard' 'Smartphone' 'Mouse' 'Tablet'
 'Webcam']

Categories:
['Accessories' 'Electronics']

Regions:
['North' 'East' 'West' 'South']

Payment Methods:
['Credit Card' 'Cash' 'Debit Card' 'UPI']

Returned:
['No' 'Yes']


In [10]:
# Pyhton Functions
def say_hello():
    print("Hello DataPulse")

In [11]:
say_hello()

Hello DataPulse


### now instead of 20 clean commands we have a function to use....


In [12]:
# Building our cleaning function
def clean_data(data):
    
    cleaned = data.copy()

    # Standardize column names
    cleaned.columns = (
        cleaned.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    # Convert date
    cleaned["order_date"] = pd.to_datetime(
        cleaned["order_date"]
    )

    # Remove duplicate records
    cleaned = cleaned.drop_duplicates()

    # Remove rows with missing critical values
    cleaned = cleaned.dropna(
        subset=[
            "order_id",
            "customer_id",
            "product"
        ]
    )

    # Keep only valid quantities
    cleaned = cleaned[
        cleaned["quantity"] > 0
    ]

    # Keep only valid prices
    cleaned = cleaned[
        cleaned["unit_price"] > 0
    ]

    # Keep valid discounts
    cleaned = cleaned[
        cleaned["discount"].between(0, 1)
    ]

    # Recalculate sales
    cleaned["gross_sales"] = (
        cleaned["quantity"] *
        cleaned["unit_price"]
    )

    cleaned["discount_amount"] = (
        cleaned["gross_sales"] *
        cleaned["discount"]
    )

    cleaned["net_sales"] = (
        cleaned["gross_sales"] -
        cleaned["discount_amount"]
    )

    return cleaned

In [13]:
# Run the pipeline
cleaned_df = clean_data(df)

In [14]:
# Compare before vs after
print("Rows before cleaning:", len(df))
print("Rows after cleaning:", len(cleaned_df))

print(
    "Rows removed:",
    len(df) - len(cleaned_df)
)

Rows before cleaning: 5000
Rows after cleaning: 5000
Rows removed: 0


In [15]:
# Check the final schema
cleaned_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   order_id         5000 non-null   object        
 1   order_date       5000 non-null   datetime64[ns]
 2   customer_id      5000 non-null   object        
 3   product          5000 non-null   object        
 4   category         5000 non-null   object        
 5   region           5000 non-null   object        
 6   quantity         5000 non-null   int64         
 7   unit_price       5000 non-null   int64         
 8   discount         5000 non-null   float64       
 9   payment_method   5000 non-null   object        
 10  returned         5000 non-null   object        
 11  gross_sales      5000 non-null   int64         
 12  discount_amount  5000 non-null   float64       
 13  net_sales        5000 non-null   float64       
dtypes: datetime64[ns](1), float64(3), int64(

In [16]:
# Adding useful date features
cleaned_df["year"] = cleaned_df["order_date"].dt.year

In [17]:
cleaned_df["month"] = cleaned_df["order_date"].dt.month

In [18]:
cleaned_df["month_name"] = (
    cleaned_df["order_date"].dt.month_name()
)

In [19]:
cleaned_df["day_name"] = (
    cleaned_df["order_date"].dt.day_name()
)

In [20]:
# Save the cleaned dataset
cleaned_df.to_csv(
    "../data/cleaned/datapulse_sales_cleaned.csv",
    index=False
)

In [21]:
# validation summary
validation_report = {
    "rows_before": len(df),
    "rows_after": len(cleaned_df),
    "duplicates": cleaned_df.duplicated().sum(),
    "missing_values": cleaned_df.isnull().sum().sum(),
    "invalid_quantities": (
        cleaned_df["quantity"] <= 0
    ).sum(),
    "invalid_prices": (
        cleaned_df["unit_price"] <= 0
    ).sum()
}

validation_report

{'rows_before': 5000,
 'rows_after': 5000,
 'duplicates': np.int64(0),
 'missing_values': np.int64(0),
 'invalid_quantities': np.int64(0),
 'invalid_prices': np.int64(0)}

# Day 7 — Data Cleaning Pipeline

## Objective

The DataPulse dataset was expanded to 5,000 orders on Day 6.
Today, a reusable Python cleaning pipeline was created to
prepare the dataset for future analysis.

## Pipeline

Raw Dataset
↓
Quality Audit
↓
Cleaning Function
↓
Data Validation
↓
Calculated Metrics
↓
Clean Dataset

## Cleaning Operations

- Standardized column names
- Converted order dates to datetime
- Removed duplicate records
- Removed records with missing critical fields
- Validated quantities
- Validated unit prices
- Validated discount values
- Recalculated sales metrics
- Added date-based features

## New Concept Learned

Python functions were introduced to make the data-cleaning
process reusable and organized.

## Key Learning

A data analyst should validate and clean data before using
it for business analysis.